In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *

In [2]:
# Load COMPAS data
features, target, scaler, feature_columns, categorical_columns, numerical_columns, one_hot_encode_features, features_ranges, features_type = load_compas()
# features = features.to_numpy()
# target = target.to_numpy()

# Train a logistic regression model
model, X_train, X_test, y_train, y_test = train_compas_model(features, target)

# Find the first negative instance from the model predictions on the test set
negative_instances = X_test[model.predict(X_test) == 0]
x = negative_instances[0]  # First negative instance

d:\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [3]:
from ugce import *

iea = UGCE(model, scaler, feature_columns, categorical_columns, \
            numerical_columns, one_hot_encode_features, features_ranges, features_type)

constraints = {
  'sex':'-',
  'age':'-',
  'race':'-',
  'c_charge_degree':'-',
}

from time import time
start = time()
best_individuals_from_scratch = iea.explain_instance(x, dynamic_constraints=False, constraints=constraints,\
                    initial_population_variability=0.9, data_distribution=True,\
                    num_generations=20, population_size=200, regeneration_tries=2,\
                    early_stopping_iterations=5,
                    num_parents=10, selection_method="sus", tournsize=20,\
                    elite_ratio=0.3, cxpb=0.8, mutpb=0.7)
end = time()
print(f"Time taken (constraints from scratch): {end-start:.2f} seconds")

Explaining instance {'sex': 1.0, 'age': 23.0, 'juv_fel_count': 0.0, 'juv_misd_count': 0.0, 'juv_other_count': 0.0, 'priors_count': 3.0, 'c_charge_degree': 0.0, 'race_0': 1.0, 'race_1': 0.0, 'race_2': 0.0, 'race_3': 0.0, 'race_4': 0.0, 'race_5': 0.0}
    Diversity: 100.00% unique individuals
Constraints: {}
Immutable features: [0, 1, 7, 8, 9, 10, 11, 12, 6]
Initial population average fitness: -9138.48487477879, max fitness: -9126.464610366815
Starting evolution...
Generation: 0
    Population size: 200, offspring size: 140
 Final population size:  200
    Average fitness: -9138.37934194668, max fitness: -9126.464610366815
Generation: 1
    Population size: 200, offspring size: 140
 Final population size:  200
    Average fitness: -9132.011551629608, max fitness: -9126.142772946785
Generation: 2
    Population size: 200, offspring size: 140
 Final population size:  200
    Average fitness: -9130.959695018691, max fitness: -9126.142772946785
Generation: 3
    Population size: 200, offspri

Best cfe is: [ 1. 23.  0.  0.  0.  0.  0.  1.  0.  0.  0.  0.  0.], guaranteed to alter the decision of the model from 0 to: 1

In [4]:
from ugce import *

iea = UGCE(model, scaler, feature_columns, categorical_columns, \
            numerical_columns, one_hot_encode_features, features_ranges, features_type)

best_individuals_dynamic = iea.explain_instance(x, dynamic_constraints=True,\
                    initial_population_variability=0.9, data_distribution=True,\
                    num_generations=20, population_size=200, population_size_dynamic=100,
                    early_stopping_iterations=5,
                    num_parents=10, selection_method="sus", tournsize=20,\
                    elite_ratio=0.3, cxpb=0.8, mutpb=0.7)

Explaining instance {'sex': 1.0, 'age': 23.0, 'juv_fel_count': 0.0, 'juv_misd_count': 0.0, 'juv_other_count': 0.0, 'priors_count': 3.0, 'c_charge_degree': 0.0, 'race_0': 1.0, 'race_1': 0.0, 'race_2': 0.0, 'race_3': 0.0, 'race_4': 0.0, 'race_5': 0.0}
    Diversity: 100.00% unique individuals
Constraints: {}
Immutable features: []
Initial population average fitness: -8417.439745989988, max fitness: 961.4971679155091
Starting evolution...
Generation: 0
    Population size: 200, offspring size: 140
 Final population size:  200
    Average fitness: -7752.107706643625, max fitness: 961.4971679155091
Generation: 1
    Population size: 200, offspring size: 140
 Final population size:  200
    Average fitness: -3787.7445752491503, max fitness: 961.4971679155091
Generation: 2
    Population size: 200, offspring size: 140
 Final population size:  200
    Average fitness: -2953.761222137523, max fitness: 961.4971679155091
Generation: 3
    Population size: 200, offspring size: 140
 Final populatio

In [5]:
iea.constraints, iea.immutables

({}, [0, 1, 6, 7, 8, 9, 10, 11, 12])

In [10]:
population = iea.population(2)
iea.fitness_assignment(population, verbose=True)
population

Feature 0 is immutable and unchanged. Reward: 100
Feature 1 is immutable and unchanged. Reward: 200
Feature 6 is immutable and unchanged. Reward: 300
Feature 7 is immutable and unchanged. Reward: 400
Feature 8 is immutable and unchanged. Reward: 500
Feature 9 is immutable and unchanged. Reward: 600
Feature 10 is immutable and unchanged. Reward: 700
Feature 11 is immutable and unchanged. Reward: 800
Feature 12 is immutable and unchanged. Reward: 900
Distance: 24.558384626353877, Sparsity: 5, Penalty: 10000, Reward: 900
Feature 0 is immutable and unchanged. Reward: 100
Feature 1 is immutable and unchanged. Reward: 200
Feature 6 is immutable and unchanged. Reward: 300
Feature 7 is immutable and unchanged. Reward: 400
Feature 8 is immutable and unchanged. Reward: 500
Feature 9 is immutable and unchanged. Reward: 600
Feature 10 is immutable and unchanged. Reward: 700
Feature 11 is immutable and unchanged. Reward: 800
Feature 12 is immutable and unchanged. Reward: 900
Distance: 36.7274247550

[Individual([1.0, 23.0, 5, 4, 1, 6, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], fitness=-9129.558384626354),
 Individual([1.0, 23.0, 19, 8, 1, 20, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], fitness=-9141.727424755081)]

In [12]:
from ugce import *

iea = UGCE(model, scaler, feature_columns, categorical_columns, \
            numerical_columns, one_hot_encode_features, features_ranges, features_type)

best_individuals = iea.explain_instance(x, dynamic_constraints=True, constraints=constraints,\
                    initial_population_variability=0.9, data_distribution=True,\
                    num_generations=20, population_size=200,  early_stopping_iterations=5,
                    num_parents=10, selection_method="rank", tournsize=20,\
                    elite_ratio=0.3, cxpb=0.8, mutpb=0.7)

Explaining instance {'sex': 1.0, 'age': 23.0, 'juv_fel_count': 0.0, 'juv_misd_count': 0.0, 'juv_other_count': 0.0, 'priors_count': 3.0, 'c_charge_degree': 0.0, 'race_0': 1.0, 'race_1': 0.0, 'race_2': 0.0, 'race_3': 0.0, 'race_4': 0.0, 'race_5': 0.0}
    Diversity: 100.00% unique individuals
Constraints: {}
Immutable features: [0, 1, 7, 8, 9, 10, 11, 12, 6]
Initial population average fitness: -9338.484874778787, max fitness: -9326.464610366815
Starting evolution...
Generation: 0
Population size: 200, offspring size: 140
Final population size:  200
Generation 0: Best fitness -9326.464610366815
    Average fitness: -9332.660304781351, max fitness: -9326.464610366815
Generation: 1
Population size: 200, offspring size: 140
Final population size:  200
Generation 1: Best fitness -9326.142772946785
    Average fitness: -9333.34300531465, max fitness: -9326.142772946785
Generation: 2
Population size: 200, offspring size: 140
Final population size:  200
Generation 2: Best fitness -9325.121157835

c:\Users\xrist\Documents\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\xrist\Documents\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\xrist\Documents\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\xrist\Documents\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\xrist\Documents\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.w

Population size: 200, offspring size: 140
Final population size:  200
Generation 7: Best fitness -9325.016208080466
    Average fitness: -9330.686113555323, max fitness: -9325.016208080466
Generation: 8
Population size: 200, offspring size: 140
Final population size:  200
Generation 8: Best fitness 1675.0639666926256
    Average fitness: -9274.68986226171, max fitness: 1675.0639666926256
Generation: 9
Population size: 200, offspring size: 140
Final population size:  200
Generation 9: Best fitness 1675.0639666926256
    Average fitness: -9055.589253944612, max fitness: 1675.0639666926256
Generation: 10
Population size: 200, offspring size: 140
Final population size:  200
Generation 10: Best fitness 1675.0639666926256
    Average fitness: -7570.312181040861, max fitness: 1675.0639666926256
Generation: 11
Population size: 200, offspring size: 140
Final population size:  200
Generation 11: Best fitness 1675.0639666926256
    Average fitness: -5151.36584808428, max fitness: 1675.06396669262

c:\Users\xrist\Documents\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [26]:
f_model(x, model), f_model(scaler.transform(np.array(best_individuals.genes).reshape(1, -1)), model)

c:\Users\xrist\Documents\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


(np.int64(0), np.int64(0))

x

In [9]:
# x_model_space, x_original_space, best_individual_model_space, best_individual_original_space
x, scaler.inverse_transform(x.reshape(1, -1)), scaler.transform(np.array(best_individuals.genes).reshape(1, -1)), best_individuals.genes

c:\Users\xrist\Documents\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


(array([1.        , 0.06410256, 0.        , 0.        , 0.        ,
        0.07894737, 0.        , 1.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ]),
 array([[ 1., 23.,  0.,  0.,  0.,  3.,  0.,  1.,  0.,  0.,  0.,  0.,  0.]]),
 array([[1.        , 0.06410256, 0.        , 0.        , 0.        ,
         0.07894737, 0.        , 1.        , 0.        , 0.        ,
         0.        , 0.        , 0.        ]]),
 [1.0, 23.0, 0.0, 0.0, 0.0, 3.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [5]:
constraints = {
  # 'sex':'-',
  # 'race':'-'
}

best_individuals = iea.explain_instance(x, dynamic_constraints=True, constraints=constraints,\
                    initial_population_variability=0.5, data_distribution=True,\
                    num_generations=20, population_size=200,  early_stopping_iterations=5,
                    num_parents=10, selection_method="tournament", tournsize=20,\
                    elite_ratio=0.3, cxpb=0.7, mutpb=0.6)

Explaining instance {'sex': 1.0, 'age': 23.0, 'juv_fel_count': 0.0, 'juv_misd_count': 0.0, 'juv_other_count': 0.0, 'priors_count': 3.0, 'c_charge_degree': 0.0, 'race_0': 1.0, 'race_1': 0.0, 'race_2': 0.0, 'race_3': 0.0, 'race_4': 0.0, 'race_5': 0.0}
    Diversity: 100.00% unique individuals
Constraints: {}
Immutable features: []
Initial population average fitness: -2019.7662763693702, max fitness: 978.1524811896237
Starting evolution...
Generation: 0
Population size: 200, offspring size: 140
Final population size:  200
Generation 0: Best fitness 978.1524811896237
    Average fitness: 191.95817835459803, max fitness: 978.1524811896237
Generation: 1
Population size: 200, offspring size: 140
Final population size:  200
Generation 1: Best fitness 978.7180699785968
    Average fitness: 33.14774513587537, max fitness: 978.7180699785968
Generation: 2
Population size: 200, offspring size: 140
Final population size:  200
Generation 2: Best fitness 978.9527661532851
    Average fitness: -462.449

In [ ]:
model.predict_proba(x.reshape(1, -1))

array([[0.60414977, 0.39585023]])

In [ ]:
model.predict_proba(np.array(best_individuals.genes).reshape(1, -1))[0]

array([0., 1.])

# Tests

In [6]:
constraints = {
  'sex':'-',
  'age':'-',
  'race':'-',
  'c_charge_degree':'-',
}

from time import time
start = time()
best_individuals = iea.explain_instance(x, dynamic_constraints=False, constraints=constraints,\
                    initial_population_variability=0.5, data_distribution=True,\
                    num_generations=20, population_size=200,  early_stopping_iterations=5,
                    num_parents=10, selection_method="tournament", tournsize=20,\
                    elite_ratio=0.3, cxpb=0.7, mutpb=0.6)
end = time()
print(f"Time taken (constraints from scratch): {end-start:.2f} seconds")


best_individuals = iea.explain_instance(x, dynamic_constraints=True, constraints=constraints,\
                    initial_population_variability=0.5, data_distribution=True,\
                    num_generations=20, population_size=200,  early_stopping_iterations=5,
                    num_parents=10, selection_method="tournament", tournsize=20,\
                    elite_ratio=0.3, cxpb=0.7, mutpb=0.6)

Explaining instance {'sex': 1.0, 'age': 23.0, 'juv_fel_count': 0.0, 'juv_misd_count': 0.0, 'juv_other_count': 0.0, 'priors_count': 3.0, 'c_charge_degree': 0.0, 'race_0': 1.0, 'race_1': 0.0, 'race_2': 0.0, 'race_3': 0.0, 'race_4': 0.0, 'race_5': 0.0}
    Diversity: 100.00% unique individuals
Constraints: {}
Immutable features: [0, 1, 7, 8, 9, 10, 11, 12, 6]
Initial population average fitness: -2843.636709757555, max fitness: 1674.8788421642328
Starting evolution...
Generation: 0
Population size: 200, offspring size: 140
Final population size:  200
Generation 0: Best fitness 1674.8788421642328
    Average fitness: 130.6482115886418, max fitness: 1674.8788421642328
Generation: 1
Population size: 200, offspring size: 140
Final population size:  200
Generation 1: Best fitness 1674.8788421642328
    Average fitness: -529.6052001929622, max fitness: 1674.8788421642328
Generation: 2
Population size: 200, offspring size: 140
Final population size:  200
Generation 2: Best fitness 1674.8788421642

AttributeError: 'list' object has no attribute 'reshape'